# Verify Fly Studio Research Output

This notebook is read-only. It accepts research_output.zip, verifies archive and artifact integrity, opens metrics and biomarkers, previews figures, and confirms the recorded pipeline status. It does not run simulation or generate data.

## Section 1 - Load the downloaded archive

Run this notebook after 20_Run_Research.ipynb. Upload the downloaded research_output.zip when Colab asks for it.

In [ ]:
from pathlib import Path
import json
import hashlib
import shutil
from zipfile import ZipFile
from pathlib import PurePosixPath

WORK_ROOT = Path.cwd().resolve()
ZIP_PATH = WORK_ROOT / 'research_output.zip'
if not ZIP_PATH.is_file():
    from google.colab import files
    uploaded = files.upload()
    if 'research_output.zip' not in uploaded:
        raise RuntimeError('Upload the exact file named research_output.zip.')
if not ZIP_PATH.is_file():
    raise RuntimeError(f'Archive not found: {ZIP_PATH}')
print(f'Input archive: {ZIP_PATH} ({ZIP_PATH.stat().st_size} bytes)')


## Section 2 - Safe extraction and integrity checks

Every archive member is checked before extraction. JSON files must parse without non-finite constants, and every dataset manifest hash is checked against the extracted file.

In [ ]:
EXTRACT_ROOT = WORK_ROOT / 'research_output_verified'
if EXTRACT_ROOT.exists():
    shutil.rmtree(EXTRACT_ROOT)
EXTRACT_ROOT.mkdir(parents=True)

with ZipFile(ZIP_PATH) as archive:
    members = archive.infolist()
    for member in members:
        relative = PurePosixPath(member.filename)
        if relative.is_absolute() or '..' in relative.parts:
            raise RuntimeError(f'Unsafe archive member: {member.filename}')
        target = (EXTRACT_ROOT / Path(*relative.parts)).resolve()
        if EXTRACT_ROOT not in target.parents and target != EXTRACT_ROOT:
            raise RuntimeError(f'Archive member escapes extraction root: {member.filename}')
    archive.extractall(EXTRACT_ROOT)

def reject_nonfinite(value):
    raise ValueError(f'Non-finite JSON constant: {value}')

json_files = sorted(EXTRACT_ROOT.rglob('*.json'))
json_documents = {}
for path in json_files:
    json_documents[path] = json.loads(path.read_text(encoding='utf-8'), parse_constant=reject_nonfinite)

for required_root in ('datasets', 'results', 'paper'):
    if not (EXTRACT_ROOT / required_root).is_dir():
        raise RuntimeError(f'Missing archive root: {required_root}/')

dataset_dirs = sorted({path.parent for path in (EXTRACT_ROOT / 'datasets').rglob('rollout.json')})
if not dataset_dirs:
    raise RuntimeError('No rollout datasets were found in the archive.')

def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

integrity_errors = []
for dataset in dataset_dirs:
    raw_candidates = [dataset / 'rollout.npz', dataset / 'rollout_arrays.npz']
    required_files = [dataset / name for name in ('rollout.json', 'manifest.json', 'metadata.json', 'viewer_pose.json')]
    required_files.append(next((path for path in raw_candidates if path.is_file()), raw_candidates[0]))
    required_files.extend([dataset / 'metrics' / 'metrics.json', dataset / 'report' / 'summary.md'])
    integrity_errors.extend(f'{path}: missing' for path in required_files if not path.is_file())
    manifest = json_documents.get(dataset / 'manifest.json')
    if not isinstance(manifest, dict) or not isinstance(manifest.get('files'), dict):
        integrity_errors.append(f'{dataset}: manifest.files missing')
        continue
    for relative_name, entry in manifest['files'].items():
        relative = PurePosixPath(str(relative_name))
        if relative.is_absolute() or '..' in relative.parts:
            integrity_errors.append(f'{dataset}: unsafe manifest path {relative_name}')
            continue
        target = dataset / Path(*relative.parts)
        expected = entry.get('sha256') if isinstance(entry, dict) else None
        if not target.is_file() or not expected or sha256(target) != expected:
            integrity_errors.append(f'{dataset}: checksum mismatch {relative_name}')
if integrity_errors:
    raise RuntimeError('Integrity validation failed:\n' + '\n'.join(integrity_errors[:20]))
print(f'Integrity: PASS ({len(dataset_dirs)} datasets, {len(json_files)} JSON documents)')


## Section 3 - Metrics

Open the metrics artifacts and show a compact table using only values present in the archive.

In [ ]:
from IPython.display import HTML, Markdown, display

metrics_paths = sorted((EXTRACT_ROOT / 'datasets').rglob('metrics/metrics.json'))
if not metrics_paths:
    raise RuntimeError('No dataset metrics.json files were found.')
metric_rows = []
for path in metrics_paths:
    payload = json.loads(path.read_text(encoding='utf-8'), parse_constant=reject_nonfinite)
    scalar = payload.get('scalar_metrics', {}) if isinstance(payload, dict) else {}
    if not isinstance(scalar, dict):
        scalar = {}
    metric_rows.append({
        'dataset_id': payload.get('dataset_id', path.parent.parent.name),
        'frame_count': payload.get('frame_count', scalar.get('frame_count')),
        'walking_speed_mm_s': payload.get('walking_speed_mm_s', scalar.get('walking_speed_mm_s')),
        'total_distance_mm': payload.get('total_distance_mm', scalar.get('total_distance_mm')),
        'trajectory_curvature': payload.get('trajectory_curvature', scalar.get('trajectory_curvature')),
    })
headers = ('dataset_id', 'frame_count', 'walking_speed_mm_s', 'total_distance_mm', 'trajectory_curvature')
rows = ['| ' + ' | '.join(headers) + ' |', '| ' + ' | '.join('---' for _ in headers) + ' |']
for row in metric_rows[:30]:
    rows.append('| ' + ' | '.join(str(row.get(header, 'unavailable')) for header in headers) + ' |')
display(Markdown('\n'.join(rows)))
if len(metric_rows) > 30:
    print(f'Showing first 30 of {len(metric_rows)} metric records.')


## Section 4 - Biomarkers

Open the biomarker reports generated by the existing pipeline. Values are displayed as computational outputs with their recorded status; no biological interpretation is added.

In [ ]:
biomarker_paths = sorted((EXTRACT_ROOT / 'results').rglob('biomarkers.json'))
if not biomarker_paths:
    raise RuntimeError('No biomarker reports were found in results/.')
biomarker_rows = []
for path in biomarker_paths:
    payload = json.loads(path.read_text(encoding='utf-8'), parse_constant=reject_nonfinite)
    values = payload.get('biomarkers', {}) if isinstance(payload, dict) else {}
    if not isinstance(values, dict):
        continue
    for name, value in values.items():
        value = value if isinstance(value, dict) else {}
        biomarker_rows.append({
            'dataset_id': payload.get('dataset_id', path.parent.name),
            'biomarker': name,
            'status': value.get('status'),
            'value': value.get('value'),
            'unit': value.get('unit'),
        })
headers = ('dataset_id', 'biomarker', 'status', 'value', 'unit')
rows = ['| ' + ' | '.join(headers) + ' |', '| ' + ' | '.join('---' for _ in headers) + ' |']
for row in biomarker_rows[:50]:
    rows.append('| ' + ' | '.join(str(row.get(header, 'unavailable')) for header in headers) + ' |')
display(Markdown('\n'.join(rows)))
print(f'Biomarker reports: {len(biomarker_paths)}; rows displayed: {min(len(biomarker_rows), 50)}')


## Section 5 - Figures

Display a representative set of generated PNG figures. No figure is regenerated.

In [ ]:
from IPython.display import Image, display

figure_paths = sorted(path for path in (EXTRACT_ROOT / 'results').rglob('*.png') if path.is_file())
figure_paths.extend(sorted(path for path in (EXTRACT_ROOT / 'datasets').rglob('figures/*.png') if path.is_file()))
figure_paths = sorted(set(figure_paths))
if not figure_paths:
    raise RuntimeError('No PNG figures were found in the archive.')
for figure_path in figure_paths[:12]:
    print(figure_path.relative_to(EXTRACT_ROOT))
    display(Image(filename=str(figure_path), width=700))


## Section 6 - Quick validation summary

In [ ]:
status_path = EXTRACT_ROOT / 'results' / 'research_status.json'
if not status_path.is_file():
    raise RuntimeError('Missing results/research_status.json.')
status_payload = json.loads(status_path.read_text(encoding='utf-8'), parse_constant=reject_nonfinite)
statuses = status_payload.get('statuses', {})
status_failures = {name: value.get('status') for name, value in statuses.items() if value.get('status') not in ('PASS', 'READY')}
required_reports = ('research_status.md', 'progress_summary.md', 'final_execution_report.md')
missing_reports = [name for name in required_reports if not (EXTRACT_ROOT / 'results' / name).is_file()]
frame_counts = []
for path in metrics_paths:
    payload = json.loads(path.read_text(encoding='utf-8'), parse_constant=reject_nonfinite)
    value = payload.get('frame_count')
    if isinstance(value, (int, float)):
        frame_counts.append(int(value))
if status_failures or missing_reports or not frame_counts:
    raise RuntimeError(f'Pipeline verification failed: statuses={status_failures}, reports={missing_reports}, frame_counts={frame_counts[:5]}')
print(f'Datasets with metrics: {len(metrics_paths)}')
print(f'Frame count range: {min(frame_counts)} - {max(frame_counts)}')
print(f'Biomarker reports: {len(biomarker_paths)}')
print(f'Figures: {len(figure_paths)}')
print('Integrity: PASS')
print('Pipeline status: PASS')
print('Research output verification completed successfully.')
